In [1]:
# ============================================================
# Project: EuroTrans Analytics
# Notebook: 04_Gold_Dimensions
# Layer: Gold
#
# Description:
# Create Gold Dimension tables from the Silver layer.
# Enrich Route Dimension with warehouse information.
# ============================================================

# ------------------------------------------------------------
# Import Libraries
# ------------------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws, col, current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType
)

# ------------------------------------------------------------
# Create Spark Session
# ------------------------------------------------------------

spark = SparkSession.builder.getOrCreate()

# ------------------------------------------------------------
# ETL Audit
# ------------------------------------------------------------

audit_log = []

print("=" * 60)
print("Starting Gold Dimension creation...")
print("=" * 60)

# ============================================================
# CUSTOMER
# ============================================================

customer = spark.table("silver_customer")

rows_before = customer.count()

(
    customer.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_customer")
)

rows_after = customer.count()

audit_log.append((
    "04_Gold_Dimensions",
    "Gold",
    "gold_dim_customer",
    rows_before,
    rows_after,
    0,
    "Success"
))

print("✓ gold_dim_customer created")

# ============================================================
# PRODUCT
# ============================================================

product = spark.table("silver_product")

rows_before = product.count()

(
    product.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_product")
)

rows_after = product.count()

audit_log.append((
    "04_Gold_Dimensions",
    "Gold",
    "gold_dim_product",
    rows_before,
    rows_after,
    0,
    "Success"
))

print("✓ gold_dim_product created")

# ============================================================
# CARRIER
# ============================================================

carrier = spark.table("silver_carrier")

rows_before = carrier.count()

(
    carrier.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_carrier")
)

rows_after = carrier.count()

audit_log.append((
    "04_Gold_Dimensions",
    "Gold",
    "gold_dim_carrier",
    rows_before,
    rows_after,
    0,
    "Success"
))

print("✓ gold_dim_carrier created")

# ============================================================
# DATE
# ============================================================

date = spark.table("silver_date")

rows_before = date.count()

(
    date.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_date")
)

rows_after = date.count()

audit_log.append((
    "04_Gold_Dimensions",
    "Gold",
    "gold_dim_date",
    rows_before,
    rows_after,
    0,
    "Success"
))

print("✓ gold_dim_date created")

# ============================================================
# WAREHOUSE
# ============================================================

warehouse = spark.table("silver_warehouse")

rows_before = warehouse.count()

(
    warehouse.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_warehouse")
)

rows_after = warehouse.count()

audit_log.append((
    "04_Gold_Dimensions",
    "Gold",
    "gold_dim_warehouse",
    rows_before,
    rows_after,
    0,
    "Success"
))

print("✓ gold_dim_warehouse created")

# ============================================================
# ROUTE
# ============================================================

route = spark.table("silver_route")

rows_before = route.count()

warehouse_origin = (
    warehouse
    .select(
        col("WarehouseID").alias("OriginWarehouseID"),
        col("WarehouseName").alias("OriginWarehouse"),
        col("City").alias("OriginCity"),
        col("Country").alias("OriginCountry")
    )
)

warehouse_destination = (
    warehouse
    .select(
        col("WarehouseID").alias("DestinationWarehouseID"),
        col("WarehouseName").alias("DestinationWarehouse"),
        col("City").alias("DestinationCity"),
        col("Country").alias("DestinationCountry")
    )
)

gold_route = (
    route
        .join(
            warehouse_origin,
            "OriginWarehouseID",
            "left"
        )
        .join(
            warehouse_destination,
            "DestinationWarehouseID",
            "left"
        )
        .withColumn(
            "RouteName",
            concat_ws(
                " → ",
                col("OriginCity"),
                col("DestinationCity")
            )
        )
)

(
    gold_route.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_route")
)

rows_after = gold_route.count()

audit_log.append((
    "04_Gold_Dimensions",
    "Gold",
    "gold_dim_route",
    rows_before,
    rows_after,
    0,
    "Success"
))

print("✓ gold_dim_route created")

# ============================================================
# COST
# ============================================================

cost_data = [
    ("Profit", 1),
    ("Fuel", 2),
    ("Driver", 3),
    ("Toll", 4),
    ("Maintenance", 5),
    ("Handling", 6)
]

schema = StructType([
    StructField("CostType", StringType(), False),
    StructField("SortOrder", IntegerType(), False)
])

gold_dim_cost = spark.createDataFrame(cost_data, schema)

rows_before = gold_dim_cost.count()

(
    gold_dim_cost.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_dim_cost")
)

rows_after = gold_dim_cost.count()

audit_log.append((
    "04_Gold_Dimensions",
    "Gold",
    "gold_dim_cost",
    rows_before,
    rows_after,
    0,
    "Success"
))

print("✓ gold_dim_cost created")

# ------------------------------------------------------------
# Write ETL Audit Log
# ------------------------------------------------------------

audit_schema = StructType([
    StructField("Notebook", StringType(), False),
    StructField("Layer", StringType(), False),
    StructField("TableName", StringType(), False),
    StructField("RowsBefore", LongType(), False),
    StructField("RowsAfter", LongType(), False),
    StructField("DuplicatesRemoved", LongType(), False),
    StructField("Status", StringType(), False)
])

audit_df = spark.createDataFrame(audit_log, audit_schema)

audit_df = audit_df.withColumn(
    "RunTimestamp",
    current_timestamp()
)

(
    audit_df.write
        .mode("append")
        .format("delta")
        .saveAsTable("etl_audit_log")
)

print("\nETL audit successfully written.")

print("=" * 60)
print("Gold Dimension Layer successfully created.")
print("=" * 60)

display(gold_dim_cost)

StatementMeta(, 0cf102b2-697d-44ba-8fef-5fb1ebc4a7eb, 3, Finished, Available, Finished, False)

Starting Gold Dimension creation...
✓ gold_dim_customer created
✓ gold_dim_product created
✓ gold_dim_carrier created
✓ gold_dim_date created
✓ gold_dim_warehouse created
✓ gold_dim_route created
✓ gold_dim_cost created

ETL audit successfully written.
Gold Dimension Layer successfully created.


SynapseWidget(Synapse.DataFrame, da9381d8-eca7-4e06-8d9b-a904942078db)